# 3교시. OCR 초안을 정돈된 데이터로 바꾸기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/03_document_structure.ipynb)

**목표:** 원문·정제 결과·변경 기록을 함께 보존합니다.

**결과물:** `clean_receipt.json`

- 기본 경로는 API 키와 OCR 모델 다운로드가 필요 없습니다.
- 선택 실습은 기본값이 `False`입니다.
- 실제 개인정보가 없는 합성 영수증만 사용합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
import json
import re

SAMPLE_OCR_TEXT = '샘플문구점\n거래일자: 2026-07-27\n연필 2개 × 1,000원 = 2,000원\n노트 1개 × 3,000원 = 3,000원\n합계: 5,000원\n'


## 핵심 3개

1. 키-값과 반복 품목은 다른 구조입니다.
2. OCR 줄 순서가 논리적 순서와 다를 수 있습니다.
3. 정제는 없는 값을 만드는 단계가 아닙니다.


In [ ]:
def normalize_line(line):
    original = line
    cleaned = re.sub(r"\s+", " ", line.strip())
    changes = []
    if original != cleaned:
        changes.append(f"공백 정리: {original!r} → {cleaned!r}")
    return cleaned, changes


def group_receipt_lines(raw_text):
    cleaned_lines = []
    change_log = []
    for raw_line in raw_text.splitlines():
        cleaned, changes = normalize_line(raw_line)
        if cleaned:
            cleaned_lines.append(cleaned)
            change_log.extend(changes)

    groups = {"header": [], "date": [], "items": [], "total": [], "other": []}
    for line in cleaned_lines:
        if "거래일자" in line:
            groups["date"].append(line)
        elif "합계" in line:
            groups["total"].append(line)
        elif "개" in line and ("×" in line or "x" in line.lower()):
            groups["items"].append(line)
        elif not groups["header"]:
            groups["header"].append(line)
        else:
            groups["other"].append(line)

    return {
        "raw_text": raw_text,
        "cleaned_lines": cleaned_lines,
        "groups": groups,
        "change_log": change_log,
    }


## 실습. 원문을 보존하며 네 영역으로 분류


In [ ]:
clean_result = group_receipt_lines(SAMPLE_OCR_TEXT)

assert clean_result["raw_text"] == SAMPLE_OCR_TEXT
assert len(clean_result["groups"]["items"]) == 2
assert clean_result["groups"]["total"] == ["합계: 5,000원"]

output_path = OUTPUT_DIR / "clean_receipt.json"
output_path.write_text(
    json.dumps(clean_result, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(clean_result["groups"], ensure_ascii=False, indent=2))
print("저장 완료:", output_path)


## mock 대체 경로

외부 `ocr_text.txt`가 없어도 내장 `SAMPLE_OCR_TEXT`를 같은 함수에 넣습니다.
정제 단계를 건너뛰지 않습니다.


## 확인

- 원문 `raw_text`가 그대로 남았는가?
- 품목이 두 줄인가?
- 원문에 없던 값을 추가하지 않았는가?
